# 02 - Random Forest

The predictive champion of the project. Random Forest
breaks past the "linear ceiling" of Logistic Regression
(+2.28 pp Acc, +3.49 pp F1 on Advanced) because it can pick
up non-linear interactions between features. What is really
interesting is that its feature-importance ranking inverts
the LR coefficient picture - see research.md section 3.5.

## 1 - Setup

Here we set matplotlib to inline mode, add the project root
to sys.path so we can import from src/, and bring in the
shared helpers together with the project-wide RANDOM_STATE = 42
(bound once in src/_constants.py, re-exported via src).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# notebooks/ sits one level under the project root, so we add
# the project root to make `from src import ...` work.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import (
    RANDOM_STATE,
    DATASETS,
    load_preprocessed,
    fit_and_score,
    build_metrics_payload,
    save_metrics,
    save_predictions,
    save_test_index,
    save_figure,
    confusion_matrix_figure,
    roc_curve_figure,
    print_dataset_block,
    print_delta,
    model_results_dir,
)

print(f"RANDOM_STATE = {RANDOM_STATE} (bound once in src/_constants.py)")

## 2 - Load the preprocessed feature matrices

load_preprocessed() gives us back the two matrices that we
made in Phase 1: the Baseline (just nutrition + tags) and
the Advanced (the Baseline plus 9 engineered culinary
features). They share the same train/test split so the A/B
comparison is fair.

In [ ]:
datasets, y_train, y_test = load_preprocessed()

print(f"Baseline X_train: {datasets['Baseline'][0].shape}")
print(f"Advanced X_train: {datasets['Advanced'][0].shape}")
print(f"y_test class balance: Miss={int((y_test == 0).sum())} / "
      f"Hit={int((y_test == 1).sum())} "
      f"(majority-class rate: {max(y_test.mean(), 1 - y_test.mean()):.4f})")

## 3 - Configure the model

200 trees gives us a small improvement over the default
100 for basically no cost, and n_jobs=-1 uses all the
cores we have. random_state=RANDOM_STATE pushes the seed
42 through to the bagging RNG.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

MODEL_SLUG    = "random_forest"
MODEL_NAME    = "RandomForest"
DISPLAY_NAME  = "Random Forest"
TOP_K         = 20

MODEL_CONFIG = {
    "n_estimators": 200,
    "n_jobs":       -1,
    "random_state": RANDOM_STATE,
}
MODEL_CONFIG

## 4 - Train on both matrices

For each matrix we build a fresh model from scratch (the
factory below gets called once per dataset). Nothing leaks
from Baseline to Advanced so the delta we see is really
only because of the 9 engineered features.

In [ ]:
def _build_model():
    return RandomForestClassifier(**MODEL_CONFIG)

per_ds_results = {}
for ds_name in DATASETS:
    X_train, X_test = datasets[ds_name]
    model = _build_model()
    result = fit_and_score(model, X_train, y_train, X_test, y_test)
    per_ds_results[ds_name] = result

    print_dataset_block(ds_name, X_train.shape, result)
    save_predictions(MODEL_SLUG, ds_name, result["y_pred"])

print_delta(per_ds_results)
save_test_index(MODEL_SLUG, y_test)

## 5 - Confusion matrix (Advanced fit)

An annotated heatmap of the confusion matrix from the
Advanced fit. We render it inline and also save it to
results/<slug>/confusion_matrix.png.

In [ ]:
adv = per_ds_results["Advanced"]
cm_array = np.array([
    [adv["confusion_matrix"]["tn"], adv["confusion_matrix"]["fp"]],
    [adv["confusion_matrix"]["fn"], adv["confusion_matrix"]["tp"]],
])
fig_cm = confusion_matrix_figure(
    cm_array,
    title=f"{DISPLAY_NAME} - Confusion Matrix (Advanced)",
)
save_figure(MODEL_SLUG, "confusion_matrix.png", fig_cm)
plt.show()

## 6 - ROC curve + AUC (Advanced fit)

We use predict_proba (or decision_function if the model
doesn't have probabilities). The AUC tells us how well the
model ranks the positives over the negatives, regardless of
the threshold we pick. Phase 4's threshold sweep is built
on top of this.

In [ ]:
fig_roc, auc = roc_curve_figure(
    y_test, adv["proba_hit"],
    title=f"{DISPLAY_NAME} - ROC Curve (Advanced)",
    model_label=DISPLAY_NAME,
)
save_figure(MODEL_SLUG, "roc_curve.png", fig_roc)
print(f"Test ROC AUC (Advanced): {auc:.4f}")
plt.show()

## 7 - Top-20 feature importances (the inversion story)

The top 7 features for RF are all continuous numerics -
three of them are engineered (`avg_words_per_step`,
`num_ingredients`, `num_steps`) and four come from the
original nutrition columns - and all six engineered has_*
keyword features place inside the top 20, above ~670 of
the editorial tags. This is the opposite of what we saw
with LR, where the binary tags were on top. The reason: L2
splits the credit across collinear features, but trees
just pick the single best splitter at each node and don't
care about collinearity. See research.md section 3.5.

In [ ]:
adv_model = per_ds_results["Advanced"]["model"]
X_train_adv = datasets["Advanced"][0]

top = (
    pd.Series(adv_model.feature_importances_, index=X_train_adv.columns)
      .sort_values(ascending=False)
      .head(TOP_K)
)
display(top.round(4).to_frame("importance"))

fig_imp, ax = plt.subplots(figsize=(9, 8))
top.iloc[::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature importance (impurity reduction, Gini)")
ax.set_title(f"Random Forest - Top {TOP_K} Feature Importances (Advanced)")
ax.grid(axis="x", alpha=0.3)
fig_imp.tight_layout()
save_figure(MODEL_SLUG, "feature_importance.png", fig_imp)
plt.show()

## 8 - Persist the canonical metrics JSON

One JSON per model, written into
results/<slug>/metrics.json. The schema is defined in
src.train_utils.build_metrics_payload, and the master
comparison notebook reads from those files.

In [ ]:
extras = {
    "top_k_feature_importances": [
        {"feature": str(name), "importance": float(val)}
        for name, val in top.items()
    ],
    "feature_importance_plot": "feature_importance.png",
    "roc_auc_advanced": auc,
}

payload = build_metrics_payload(
    model_name=MODEL_NAME,
    display_name=DISPLAY_NAME,
    model_config=MODEL_CONFIG,
    n_train=len(y_train),
    n_test=len(y_test),
    random_state=RANDOM_STATE,
    per_dataset_results=per_ds_results,
    extras=extras,
)
metrics_path = save_metrics(MODEL_SLUG, payload)
print(f"Wrote {metrics_path.relative_to(metrics_path.parent.parent.parent)}")

## 9 - Summary

    **Model:** Random Forest (n=200)

    - **Test Accuracy / F1:** about 0.6239 / 0.6753 - the top
  scores in the whole lineup.
- **Breaks the linear ceiling** by +2.28 pp Acc and +3.49 pp F1
  over LR.
- **Inverts the feature-importance ranking** - the continuous
  numerics dominate (and all six has_* features crack the top
  20) where they sat in the bottom half of the LR ranking.

    To see this model side by side with the other six, run the
    master comparison notebook (`08_Master_Comparison.ipynb`).